In [164]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [165]:
df1 = pd.read_csv("D:\\CHROME DOWNLOADS\\Road_Accident_PRedictio_Datasets\\Road_Accident_Severity_Project\\accident_data_2008_2020.csv")
df = df1.copy()
df.drop(columns=['accident_index','location_easting_osgr','lsoa_of_accident_location','location_northing_osgr', 'accident_reference', 'local_authority_ons_district', 'local_authority_highway'], errors='ignore', inplace=True)
df.head()

C:\Users\krish\AppData\Local\Temp\ipykernel_12488\1560869173.py:1: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv("D:\\CHROME DOWNLOADS\\Road_Accident_PRedictio_Datasets\\Road_Accident_Severity_Project\\accident_data_2008_2020.csv")


,accident_year,longitude,latitude,police_force,accident_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,...,pedestrian_crossing_human_control,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag
0,2008,-0.214677,51.506812,1,2,1,1,23/01/2008,4,19:46,...,0,0,4,1,1,0,0,1,1,2
1,2008,-0.170138,51.496323,1,2,1,1,15/02/2008,6,12:53,...,0,4,1,1,1,0,0,1,1,2
2,2008,-0.190946,51.502042,1,2,1,1,27/02/2008,4,21:50,...,0,5,4,1,1,0,0,1,1,2
3,2008,-0.193763,51.492733,1,2,1,1,25/02/2008,2,13:55,...,0,4,1,1,1,0,0,1,1,2
4,2008,-0.199504,51.493271,1,2,2,1,27/02/2008,4,19:25,...,0,0,4,1,1,0,0,1,1,2


In [116]:
df.shape

(1808615, 29)

In [167]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1808615 entries, 0 to 1808614
Data columns (total 29 columns):
 #   Column                                       Dtype  
---  ------                                       -----  
 0   accident_year                                int64  
 1   longitude                                    float64
 2   latitude                                     float64
 3   police_force                                 int64  
 4   accident_severity                            int64  
 5   number_of_vehicles                           int64  
 6   number_of_casualties                         int64  
 7   date                                         object 
 8   day_of_week                                  int64  
 9   time                                         object 
 10  local_authority_district                     int64  
 11  first_road_class                             int64  
 12  first_road_number                            int64  
 13  road_type   

In [118]:
df.columns

Index(['accident_year', 'longitude', 'latitude', 'police_force',
       'accident_severity', 'number_of_vehicles', 'number_of_casualties',
       'date', 'day_of_week', 'time', 'local_authority_district',
       'first_road_class', 'first_road_number', 'road_type', 'speed_limit',
       'junction_detail', 'junction_control', 'second_road_class',
       'second_road_number', 'pedestrian_crossing_human_control',
       'pedestrian_crossing_physical_facilities', 'light_conditions',
       'weather_conditions', 'road_surface_conditions',
       'special_conditions_at_site', 'carriageway_hazards',
       'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident',
       'trunk_road_flag'],
      dtype='object')

In [169]:
df_float64 = df[['longitude','latitude','police_force']]
df_int64 = df[['accident_year','police_force','accident_severity','number_of_vehicles','number_of_casualties','day_of_week','local_authority_district','first_road_class','first_road_number','road_type','speed_limit','junction_detail','junction_control','second_road_class','second_road_number','light_conditions','weather_conditions','road_surface_conditions','urban_or_rural_area','did_police_officer_attend_scene_of_accident']]
df_todatetime = df[['date','time']]


def conversion(df):
    floating_32 = df.astype('float32')
    return floating_32

def converted(df):
    integer_32 = df.astype('int32',errors='ignore')
    return integer_32

df['accident_datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce',dayfirst=True)
df['accident_month'] = df['accident_datetime'].dt.month
df['accident_hour'] = df['accident_datetime'].dt.hour
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if (x >= 5) & (x <= 7) else 0)
df = df.drop(columns=['date','time'], errors='ignore')

df_float64 = conversion(df_float64)
df_int64 = converted(df_int64)
df[df_float64.columns] = df_float64
df[df_int64.columns] = df_int64

In [120]:
missing_summary = df[['latitude', 'longitude']].isnull().sum()
print("Missing Coordinates:\n", missing_summary)
print("Percentage of missing coordinates:", missing_summary.sum() / len(df) * 100)
print('-----------')
df = df.dropna(subset=['latitude', 'longitude'])
print('Dropped Null Coordinates:', missing_summary.sum() - df[['latitude', 'longitude']].isnull().sum().sum())

Missing Coordinates:
 latitude     170
longitude    170
dtype: int64
Percentage of missing coordinates: 0.01879891519201157
-----------
Dropped Null Coordinates: 340


In [121]:
string_cols = ['police_force', 'first_road_number', 'second_road_number', 'local_authority_district']
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str)

In [122]:
df['speed_limit'].unique()

array([30., 40., 50., 60., 15., 20., 70., 10.,  0., nan, -1.])

In [170]:
def assign_time_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['time_of_day'] = df['accident_hour'].apply(assign_time_of_day)

In [171]:
df['time_of_day'].unique()

array(['Evening', 'Afternoon', 'Night', 'Morning'], dtype=object)

In [124]:
df.shape

(1808445, 32)

In [125]:
df.columns

Index(['accident_year', 'longitude', 'latitude', 'police_force',
       'accident_severity', 'number_of_vehicles', 'number_of_casualties',
       'day_of_week', 'local_authority_district', 'first_road_class',
       'first_road_number', 'road_type', 'speed_limit', 'junction_detail',
       'junction_control', 'second_road_class', 'second_road_number',
       'pedestrian_crossing_human_control',
       'pedestrian_crossing_physical_facilities', 'light_conditions',
       'weather_conditions', 'road_surface_conditions',
       'special_conditions_at_site', 'carriageway_hazards',
       'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident',
       'trunk_road_flag', 'accident_datetime', 'accident_month',
       'accident_hour', 'is_weekend', 'time_of_day'],
      dtype='object')

In [126]:
common_speed = df['speed_limit'].mode()[0]
df.iloc[:,14] = common_speed

In [127]:
filtered_speed = df.loc[df['speed_limit'] < 10, 'speed_limit']
print(filtered_speed.count())

def fill_with_mode(group):
    mode_series = group.mode()
    if not mode_series.empty:
        return group.fillna(mode_series.iloc[0])
    return group

df['speed_limit'] = df.groupby(['urban_or_rural_area', 'first_road_class'])['speed_limit'].transform(fill_with_mode)


df['speed_limit'] = df['speed_limit'].fillna(df['speed_limit'].median())


93


In [128]:
df.isnull().sum()

accident_year                                  0
longitude                                      0
latitude                                       0
police_force                                   0
accident_severity                              0
number_of_vehicles                             0
number_of_casualties                           0
day_of_week                                    0
local_authority_district                       0
first_road_class                               0
first_road_number                              0
road_type                                      0
speed_limit                                    0
junction_detail                                0
junction_control                               0
second_road_class                              0
second_road_number                             0
pedestrian_crossing_human_control              0
pedestrian_crossing_physical_facilities        0
light_conditions                               0
weather_conditions  

In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1808445 entries, 0 to 1808614
Data columns (total 32 columns):
 #   Column                                       Dtype         
---  ------                                       -----         
 0   accident_year                                int32         
 1   longitude                                    float32       
 2   latitude                                     float32       
 3   police_force                                 object        
 4   accident_severity                            int32         
 5   number_of_vehicles                           int32         
 6   number_of_casualties                         int32         
 7   day_of_week                                  int32         
 8   local_authority_district                     object        
 9   first_road_class                             int32         
 10  first_road_number                            object        
 11  road_type                                 

In [130]:
def duplicates(df):
    summary_data = []
    for col in df.columns:
        dup_count = df[col].duplicated().sum()
        dup_percentage = dup_count / len(df) * 100

        summary_data.append({
            "Column Name": col,
            "Duplicate Count": dup_count,
            "Duplicate Percentage": f"{dup_percentage:.2f}%"
        })
    return pd.DataFrame(summary_data)

duplicates(df)

,Column Name,Duplicate Count,Duplicate Percentage
0,accident_year,1808432,100.00%
1,longitude,460592,25.47%
2,latitude,1026594,56.77%
3,police_force,1808393,100.00%
4,accident_severity,1808442,100.00%
5,number_of_vehicles,1808418,100.00%
6,number_of_casualties,1808392,100.00%
7,day_of_week,1808438,100.00%
8,local_authority_district,1808028,99.98%
9,first_road_class,1808439,100.00%


In [131]:
print("Total duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates(keep='first')
df.reset_index(drop=True, inplace=True)
print("Total duplicate rows:", df.duplicated().sum())

Total duplicate rows: 39
Total duplicate rows: 0


In [132]:
df.duplicated().sum()

np.int64(0)

In [133]:
total_rows = len(df)
true_duplicates = df.duplicated().sum()
duplicate_ratio = (true_duplicates / total_rows) * 100

print(f"Total Rows in Dataset: {total_rows:,}")
print(f"True Row-level Duplicates: {true_duplicates:,} ({duplicate_ratio:.2f}%)")


Total Rows in Dataset: 1,808,406
True Row-level Duplicates: 0 (0.00%)


In [134]:
df['speed_limit'].unique()

array([30., 40., 50., 60., 15., 20., 70., 10.,  0., -1.])

In [135]:
df['speed_limit'] = pd.to_numeric(df['speed_limit'], errors='coerce')

df['speed_limit'] = df.groupby(['urban_or_rural_area', 'road_type'])['speed_limit'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 30)
)

df.loc[df['speed_limit'].isnull() & (df['urban_or_rural_area'] == 'Urban'), 'speed_limit'] = 30
df.loc[df['speed_limit'].isnull() & (df['urban_or_rural_area'] == 'Rural'), 'speed_limit'] = 60

df['speed_limit'] = df['speed_limit'].fillna(df['speed_limit'].mode()[0])

print("Remaining nulls in speed_limit:", df['speed_limit'].isnull().sum())

Remaining nulls in speed_limit: 0


In [136]:
df['number_of_casualties'].value_counts()

number_of_casualties
1     1409192
2      275840
3       78645
4       28101
5       10158
6        3774
7        1317
8         561
9         270
10        172
11         94
12         65
13         42
14         27
15         16
16         16
17         16
19         13
21         10
18          8
27          6
24          5
26          5
25          4
22          4
36          3
42          3
23          3
29          3
20          3
43          2
34          2
87          2
41          2
51          2
33          2
38          2
93          1
54          1
35          1
70          1
59          1
58          1
48          1
46          1
63          1
40          1
28          1
47          1
45          1
62          1
32          1
52          1
Name: count, dtype: int64

In [137]:
def casualities(num_cas):
    if num_cas >=1 and num_cas <2:
        return "1"
    elif num_cas >=2 and num_cas <3:
        return "2"
    elif num_cas >=3 and num_cas <4:
        return "3"
    elif num_cas >= 4 and num_cas <5:
        return "4"
    elif num_cas >= 5:
        return "5+"

df['number_of_casualties']= df['number_of_casualties'].apply(casualities)

In [138]:
df['number_of_casualties'].value_counts()

number_of_casualties
1     1409192
2      275840
3       78645
4       28101
5+      16628
Name: count, dtype: int64

In [139]:
print(df.nunique().sort_values(ascending=False))

longitude                                      1347853
accident_datetime                              1202677
latitude                                        781851
first_road_number                                 3968
second_road_number                                3759
local_authority_district                           417
police_force                                        52
number_of_vehicles                                  27
accident_hour                                       24
accident_year                                       13
accident_month                                      12
junction_detail                                     11
speed_limit                                         10
special_conditions_at_site                          10
weather_conditions                                  10
second_road_class                                    9
carriageway_hazards                                  8
pedestrian_crossing_physical_facilities              8
road_surfa

In [140]:
df['number_of_vehicles'].value_counts()/df.shape[0]*100

number_of_vehicles
2     60.046085
1     29.946151
3      7.746656
4      1.684301
5      0.379727
6      0.115571
7      0.043906
8      0.019409
9      0.008405
10     0.004424
11     0.001770
12     0.000885
13     0.000829
14     0.000553
16     0.000498
19     0.000111
18     0.000111
15     0.000111
32     0.000055
29     0.000055
34     0.000055
67     0.000055
21     0.000055
37     0.000055
23     0.000055
24     0.000055
17     0.000055
Name: count, dtype: float64

In [141]:
def vehicles(num_veh):
    if num_veh >=1 and num_veh <2:
        return "1"
    elif num_veh >=2 and num_veh <3:
        return "2"
    elif num_veh >=3 and num_veh <4:
        return "3"
    elif num_veh >= 4:
        return "4+"


df['number_of_vehicles']= df['number_of_vehicles'].apply(vehicles)

In [142]:
df['number_of_vehicles'].value_counts()/df.shape[0]*100

number_of_vehicles
2     60.046085
1     29.946151
3      7.746656
4+     2.261107
Name: count, dtype: float64

In [143]:
print(df['number_of_vehicles'].dtypes)
df['number_of_vehicles']=df['number_of_vehicles'].astype('object')

object


In [144]:
def getSeason(month):
    if (month == 2 or month == 12 or month == 1):
       return "Winter"
    elif(month == 6 or month == 7 or month ==8 or month == 9):
       return "Rainy"
    elif(month == 3 or month== 4 or month == 5):
       return "Summer"
    else:
       return "Autumn"

df['season'] = df['accident_month'].apply(getSeason)

In [145]:
df['season'].value_counts()/df.shape[0]*100

season
Rainy     34.203658
Winter    23.889934
Summer    23.750032
Autumn    18.156376
Name: count, dtype: float64

In [146]:
df['accident_severity'].value_counts()/len(df) * 100
# 1 -> Fatal
# 2 -> Serious
# 3 -> Slight

accident_severity
3    83.505363
2    15.248512
1     1.246125
Name: count, dtype: float64

In [147]:
df['accident_seriousness'] = df['accident_severity']
df['accident_seriousness'] = df['accident_seriousness'].replace(to_replace=3,value="Not Serious")
df['accident_seriousness'] = df['accident_seriousness'].replace(to_replace=2,value="Serious")
df['accident_seriousness'] = df['accident_seriousness'].replace(to_replace=1,value="Fatal")

In [148]:
df['accident_seriousness'].value_counts()/df.shape[0]*100

accident_seriousness
Not Serious    83.505363
Serious        15.248512
Fatal           1.246125
Name: count, dtype: float64

In [149]:
df.accident_seriousness.value_counts()

accident_seriousness
Not Serious    1510116
Serious         275755
Fatal            22535
Name: count, dtype: int64

In [150]:
df['weather_conditions'].value_counts()/df.shape[0]*100
# Fine no high winds = 1
# Raining no high winds = 2
# Other = 9
# Fine + high winds = 8
# Snowing no high winds = 5
# Raining + high winds = 4
# Unknown = 3
# Fog or mist = 7 
# Snowing + high winds = 6

weather_conditions
 1    80.077538
 2    11.629026
 9     2.248500
 8     2.152890
 5     1.364185
 4     1.218421
 3     0.664342
 7     0.499832
 6     0.135700
-1     0.009566
Name: count, dtype: float64

In [151]:
df.drop(df[df['weather_conditions'] == -1].index, inplace=True)

In [152]:
df['weather_conditions'].value_counts()/df.shape[0]*100

weather_conditions
1    80.085199
2    11.630138
9     2.248715
8     2.153096
5     1.364315
4     1.218538
3     0.664406
7     0.499880
6     0.135713
Name: count, dtype: float64

In [153]:
df['light_conditions'].value_counts()/df.shape[0]*100

light_conditions
 1    72.886514
 4    19.791255
 6     5.279298
 7     1.493392
 5     0.549431
-1     0.000111
Name: count, dtype: float64

In [154]:
df.drop(df[df['light_conditions'] == -1].index, inplace=True)

In [155]:
perct_val = df['light_conditions'].value_counts()/df.shape[0]*100
perct_val.sort_values(ascending=False)
# Daylight               
# Darkness - lights lit      
# Darkness - no lighting     
# Darkness - lights unlit 

light_conditions
1    72.886595
4    19.791277
6     5.279303
7     1.493393
5     0.549432
Name: count, dtype: float64

In [156]:
df['season'].value_counts()/df.shape[0]*100

season
Rainy     34.202986
Winter    23.890366
Summer    23.750284
Autumn    18.156364
Name: count, dtype: float64

In [157]:
df.drop(df[df['speed_limit'] == -1.0].index, inplace=True)
df['speed_limit'].value_counts()/df.shape[0]*100

speed_limit
30.0    63.185629
60.0    14.201342
40.0     8.304229
70.0     6.774534
20.0     3.808612
50.0     3.724548
10.0     0.000553
15.0     0.000498
0.0      0.000055
Name: count, dtype: float64

In [158]:
df['day_of_week'].value_counts()/df.shape[0]*100

day_of_week
6    16.318989
5    15.135562
4    15.068698
3    15.017872
2    14.233585
7    13.241847
1    10.983448
Name: count, dtype: float64

In [159]:
df.drop(df[df['road_surface_conditions'] == -1].index, inplace=True)
df['road_surface_conditions'].value_counts()/df.shape[0]*100

road_surface_conditions
1    70.018575
2    26.958136
4     1.981632
3     0.640687
9     0.257616
5     0.143354
Name: count, dtype: float64

In [160]:
df['accident_seriousness'].value_counts()/df.shape[0]*100

accident_seriousness
Not Serious    83.492295
Serious        15.260243
Fatal           1.247462
Name: count, dtype: float64

In [161]:
df.shape

(1804624, 34)

In [162]:
df.isnull().sum()

accident_year                                  0
longitude                                      0
latitude                                       0
police_force                                   0
accident_severity                              0
number_of_vehicles                             0
number_of_casualties                           0
day_of_week                                    0
local_authority_district                       0
first_road_class                               0
first_road_number                              0
road_type                                      0
speed_limit                                    0
junction_detail                                0
junction_control                               0
second_road_class                              0
second_road_number                             0
pedestrian_crossing_human_control              0
pedestrian_crossing_physical_facilities        0
light_conditions                               0
weather_conditions  

In [163]:
df.to_csv('Cleaned_Accident_data_2008_2020.csv', index=False)